# DrowsyGuard — huấn luyện CNN miệng

Notebook đọc video YawDD đã tải sẵn trong Google Drive, trích crop miệng bằng MediaPipe và không tải dataset. Đặt project tại `MyDrive/DrowsyGuard`; model được xuất vào `models/`. Cấu hình token nhãn và hàm tìm subject theo đúng cấu trúc dataset của bạn trước khi chạy. Notebook dừng với lỗi nếu không xác định duy nhất nhãn hoặc subject — không chia ngẫu nhiên theo frame.

Nhãn: `0 = không ngáp`, `1 = ngáp`. Các frame cùng subject luôn nằm trong cùng một split.

In [ ]:
%pip -q install mediapipe==0.10.21
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import sys
PROJECT_ROOT = Path('/content/drive/MyDrive/DrowsyGuard')
if not (PROJECT_ROOT / 'ml' / 'colab_training.py').is_file():
    raise FileNotFoundError('Hãy đặt/copy project DrowsyGuard vào MyDrive/DrowsyGuard.')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.colab_training import (
    extract_yawdd_mouth_crops, make_tf_dataset, report_binary_metrics,
    split_by_subject, train_mobilenet_binary, export_tflite,
)

In [ ]:
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/YawDD')
CROPS_ROOT = PROJECT_ROOT / 'ml' / 'generated_mouth_crops'
SAMPLE_FPS = 2.0

# Chỉnh các token theo tên thư mục/video trong bản YawDD của bạn.
POSITIVE_PATH_TOKENS = ('yawning', 'yawn')
NEGATIVE_PATH_TOKENS = ('non-yawn', 'nonyawn', 'no-yawn', 'normal')
SUBJECT_PATTERN = re.compile(r'(?:subject|driver|person|participant|s|p)[_-]?(\d+)', re.IGNORECASE)

def parse_video_label(video_path, dataset_root):
    text = '/'.join(video_path.relative_to(dataset_root).parts).lower().replace('_', '-')
    if any(token in text for token in NEGATIVE_PATH_TOKENS):
        return 0
    if any(token in text for token in POSITIVE_PATH_TOKENS):
        return 1
    raise ValueError(f'Chưa map được nhãn video: {video_path}; hãy chỉnh POSITIVE/NEGATIVE_PATH_TOKENS.')

def parse_video_subject(video_path, dataset_root):
    relative = video_path.relative_to(dataset_root)
    for part in reversed(relative.parts):
        match = SUBJECT_PATTERN.search(part)
        if match:
            return match.group(0).lower()
    for part in relative.parts[:-1]:
        if part.isdigit():
            return part
    raise ValueError(f'Chưa tìm được subject ID: {video_path}; hãy chỉnh parse_video_subject.')

records = extract_yawdd_mouth_crops(
    DATASET_ROOT,
    CROPS_ROOT,
    label_parser=parse_video_label,
    subject_parser=parse_video_subject,
    sample_fps=SAMPLE_FPS,
)
print(f'{len(records)} mouth crops; {len({item.subject_id for item in records})} subjects')
print(f'Không ngáp: {sum(item.label == 0 for item in records)}; ngáp: {sum(item.label == 1 for item in records)}')
train_records, validation_records, test_records = split_by_subject(records, seed=42)
for name, split in [('train', train_records), ('validation', validation_records), ('test', test_records)]:
    print(f'{name}: {len(split)} crops, {len({item.subject_id for item in split})} subjects')
assert not ({x.subject_id for x in train_records} & {x.subject_id for x in validation_records})
assert not ({x.subject_id for x in train_records} & {x.subject_id for x in test_records})
assert not ({x.subject_id for x in validation_records} & {x.subject_id for x in test_records})

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS_PER_STAGE = 20
train_ds = make_tf_dataset(train_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=True)
validation_ds = make_tf_dataset(validation_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=False)
test_ds = make_tf_dataset(test_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=False)

model = train_mobilenet_binary(
    train_ds, validation_ds, image_size=IMAGE_SIZE, epochs_per_stage=EPOCHS_PER_STAGE
)
confusion_matrix, roc_auc = report_binary_metrics(model, test_ds)
print('Test ROC-AUC (đo trên dataset đang dùng):', roc_auc)
export_tflite(model, PROJECT_ROOT / 'models' / 'mouth_classifier.tflite')